In [43]:
import os
from datetime import datetime
from time import perf_counter

import kagglehub
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import plotly.express as px
from plotly.subplots import make_subplots



In [44]:
print("Downloading Fashion MNIST dataset...")
dataset_path = kagglehub.dataset_download("zalando-research/fashionmnist")
print(f"Dataset downloaded to: {dataset_path}")



Dataset downloaded to: /Users/andreaigner/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4


In [45]:
train_data = pd.read_csv(os.path.join(dataset_path, "fashion-mnist_train.csv"))
print(f"Training data shape: {train_data.shape}")
train_data.head()



Training data shape: (60000, 785)


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
labels = train_data["label"].to_numpy()
features = train_data.drop(columns=["label"]).to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
print(f"Scaled feature matrix shape: {X_scaled.shape}")



Scaled feature matrix shape: (60000, 784)


In [47]:
pca_components = 50
pca_whiten = PCA(n_components=pca_components, whiten=True, random_state=42)

pca_start_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pca_start = perf_counter()
print(f"[{pca_start_ts}] Fitting PCA with whiten=True and {pca_components} components...")
X_pca50 = pca_whiten.fit_transform(X_scaled)
pca_duration = perf_counter() - pca_start
pca_end_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"[{pca_end_ts}] Finished PCA in {pca_duration:.2f} seconds")
print(f"PCA output shape: {X_pca50.shape}")



[2025-11-11 14:43:38] Fitting PCA with whiten=True and 50 components...
[2025-11-11 14:43:38] Finished PCA in 0.16 seconds
PCA output shape: (60000, 50)


In [48]:
rng = np.random.default_rng(42)
sample_size = 6000
sample_idx = rng.choice(X_pca50.shape[0], size=sample_size, replace=False)

X_sample = X_pca50[sample_idx]
labels_sample = labels[sample_idx]
print(f"Sampled subset shape: {X_sample.shape}")



Sampled subset shape: (6000, 50)


In [49]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
label_to_class = {idx: name for idx, name in enumerate(class_names)}
class_labels = [label_to_class[int(lbl)] for lbl in labels_sample]



In [50]:
def run_tsne(data, perplexity, early_exaggeration, learning_rate):
    tsne = TSNE(
        n_components=3,
        perplexity=perplexity,
        early_exaggeration=early_exaggeration,
        learning_rate=learning_rate,
        init="pca",
        random_state=42,
    )
    start = perf_counter()
    embedding = tsne.fit_transform(data)
    duration = perf_counter() - start
    return embedding, duration



In [51]:
tsne_settings = [
    {"perplexity": 20, "early_exaggeration": 12.0, "learning_rate": 200},
    {"perplexity": 35, "early_exaggeration": 16.0, "learning_rate": 300},
    {"perplexity": 50, "early_exaggeration": 24.0, "learning_rate": 400},
]

embeddings = []
for config in tsne_settings:
    params_str = (
        f"perp={config['perplexity']}, exag={config['early_exaggeration']}, lr={config['learning_rate']}"
    )
    print(f"Running t-SNE with {params_str}...")
    emb, duration = run_tsne(
        X_sample,
        perplexity=config["perplexity"],
        early_exaggeration=config["early_exaggeration"],
        learning_rate=config["learning_rate"],
    )
    print(f"  Completed in {duration:.2f} seconds")
    embeddings.append((params_str, emb, duration))



Running t-SNE with perp=20, exag=12.0, lr=200...
  Completed in 21.43 seconds
Running t-SNE with perp=35, exag=16.0, lr=300...
  Completed in 24.93 seconds
Running t-SNE with perp=50, exag=24.0, lr=400...
  Completed in 46.76 seconds


In [52]:
fig = make_subplots(
    rows=1,
    cols=len(embeddings),
    specs=[[{"type": "scene"} for _ in embeddings]],
    subplot_titles=[title for title, _, _ in embeddings],
)

for idx, (title, values, duration) in enumerate(embeddings):
    emb_df = pd.DataFrame({
        "Dim1": values[:, 0],
        "Dim2": values[:, 1],
        "Dim3": values[:, 2],
        "label": labels_sample,
        "Class": class_labels,
    })
    scatter = px.scatter_3d(
        emb_df,
        x="Dim1",
        y="Dim2",
        z="Dim3",
        color="Class",
        hover_data={"label": True, "Class": True},
    )
    for trace in scatter.data:
        if idx > 0:
            trace.showlegend = False
        fig.add_trace(trace, row=1, col=idx + 1)
    fig.layout[f"scene{'' if idx == 0 else idx + 1}"].xaxis.title = "Component 1"
    fig.layout[f"scene{'' if idx == 0 else idx + 1}"].yaxis.title = "Component 2"
    fig.layout[f"scene{'' if idx == 0 else idx + 1}"].zaxis.title = "Component 3"

base_embeddings_3d = [embeddings[i][1] for i in range(min(2, len(embeddings)))]
if base_embeddings_3d:
    dim_ranges = []
    for axis in range(3):
        axis_vals = np.concatenate([arr[:, axis] for arr in base_embeddings_3d])
        axis_min = axis_vals.min()
        axis_max = axis_vals.max()
        span = axis_max - axis_min
        padding = 0.05 * span if span > 0 else 1.0
        dim_ranges.append((axis_min - padding, axis_max + padding))
    scene_names = [f"scene{'' if i == 0 else i + 1}" for i in range(len(embeddings))]
    for scene_name in scene_names:
        scene = fig.layout[scene_name]
        scene.xaxis.range = dim_ranges[0]
        scene.yaxis.range = dim_ranges[1]
        scene.zaxis.range = dim_ranges[2]

fig.update_layout(
    height=600,
    width=380 * len(embeddings),
    title_text=f"3D t-SNE Variants on PCA-whitened ({pca_components} comps) sample of {sample_size}",
    legend=dict(title="Class", itemsizing="constant"),
)

fig.show()



In [53]:
fig_2d = make_subplots(
    rows=1,
    cols=len(embeddings),
    subplot_titles=[title for title, _, _ in embeddings],
)

for idx, (title, values, duration) in enumerate(embeddings):
    emb_df = pd.DataFrame({
        "Dim1": values[:, 0],
        "Dim2": values[:, 1],
        "label": labels_sample,
        "Class": class_labels,
    })
    scatter2d = px.scatter(
        emb_df,
        x="Dim1",
        y="Dim2",
        color="Class",
        hover_data={"label": True, "Class": True},
    )
    for trace in scatter2d.data:
        if idx > 0:
            trace.showlegend = False
        fig_2d.add_trace(trace, row=1, col=idx + 1)
    fig_2d.update_xaxes(title_text="Component 1", row=1, col=idx + 1)
    fig_2d.update_yaxes(title_text="Component 2", row=1, col=idx + 1)

base_embeddings_2d = [embeddings[i][1][:, :2] for i in range(min(2, len(embeddings)))]
if base_embeddings_2d:
    ranges_2d = []
    for axis in range(2):
        axis_vals = np.concatenate([arr[:, axis] for arr in base_embeddings_2d])
        axis_min = axis_vals.min()
        axis_max = axis_vals.max()
        span = axis_max - axis_min
        padding = 0.05 * span if span > 0 else 1.0
        ranges_2d.append((axis_min - padding, axis_max + padding))
    for idx in range(len(embeddings)):
        fig_2d.layout[f"xaxis{'' if idx == 0 else idx + 1}"].update(range=ranges_2d[0])
        fig_2d.layout[f"yaxis{'' if idx == 0 else idx + 1}"].update(range=ranges_2d[1])

fig_2d.update_layout(
    height=500,
    width=350 * len(embeddings),
    title_text=f"2D Projections of 3D t-SNE Variants (first two dims)",
    legend=dict(title="Class", itemsizing="constant"),
)

fig_2d.show()



In [54]:
results_df = pd.DataFrame(
    [
        {
            "Config": title,
            "Duration (s)": duration,
        }
        for title, _, duration in embeddings
    ]
)
results_df



,Config,Duration (s)
0,"perp=20, exag=12.0, lr=200",21.430784
1,"perp=35, exag=16.0, lr=300",24.934110
2,"perp=50, exag=24.0, lr=400",46.758785
